# Entraînement et expérimentation

Ce notebook permet d'entraîner et d'évaluer les modèles de diagnostic et de recommandation.


In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

from src.data_preparation import prepare_diagnostic_data
from src.train_diagnostic import compare_models
from src.train_recommender import train_recommender
from src.utils import load_experiment_results


## Préparation des données


In [ ]:
# Prepare data
data = prepare_diagnostic_data(
    data_path="../data/raw/symptoms_disease.csv",
    test_size=0.2
)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


## Entraînement du modèle de diagnostic


In [ ]:
# Compare models
results = compare_models(X_train, y_train, X_test, y_test)

# Display results
for model_name, result in results.items():
    if model_name != "best_model":
        print(f"\n{model_name.upper()}:")
        print(f"  Test F1-Score: {result['metrics']['test']['f1_score']:.4f}")
        print(f"  Test Accuracy: {result['metrics']['test']['accuracy']:.4f}")


## Entraînement du système de recommandation


In [ ]:
# Train recommender
recommender = train_recommender(
    data_path="../data/raw/symptoms_disease.csv",
    n_clusters=5,
    use_pca=False
)

# Test recommendations
if len(recommender.case_features) > 0:
    sample_case = recommender.case_features.index[0]
    recommendations = recommender.recommend_next_cases(sample_case, n_recommendations=3)
    print(f"\nExemple de recommandations pour '{sample_case}':")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")


## Chargement des résultats


In [ ]:
# Load experiment results
try:
    results = load_experiment_results("../experiments/experiment_results.json")
    print("Résultats d'expérimentation:")
    print(results)
except FileNotFoundError:
    print("Aucun résultat d'expérimentation trouvé.")
